# LinkedIn MCP

In [ ]:
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
load_dotenv()

In [ ]:
# Connect to LinkedIn MCP
# Run the MCP server from its own directory so uv finds the project (avoids "Connection closed").
# Prerequisites (from linkedin-mcp-server dir):
#   uv run linkedin-mcp-server --get-session   # create LinkedIn session once
#   uv run playwright install chromium         # if you see "Failed to start browser"

uv_path = shutil.which("uv") or "/opt/homebrew/bin/uv"
linkedin_server_dir = os.path.expanduser("~/dev/linkedin-mcp-server")
env = os.environ.copy()
if "/opt/homebrew/bin" not in env.get("PATH", ""):
    env["PATH"] = f"/opt/homebrew/bin:{env.get('PATH', '')}"
env["TRANSPORT"] = "stdio"

client = MultiServerMCPClient(
    {
        "linkedin": {
            "command": uv_path,
            "transport": "stdio",
            "args": ["--directory", linkedin_server_dir, "run", "linkedin-mcp-server"],
            "cwd": linkedin_server_dir,
            "env": env,
        }
    }
)
tools = await client.get_tools()

In [ ]:
tools

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="you are a helpful assistant that can answer questions about the companies on linked in",
)
# MCP tools are async-only; use ainvoke so the graph runs tools asynchronously
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the directors of company Goodmans Consulting")]}
)
result

In [ ]:
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the activities or services of Goodmans-Consulting")]}
)
result